In [25]:
from docling.document_converter import DocumentConverter
import fitz  # PyMuPDF
import os
import glob
from datetime import datetime


In [9]:
def clean_pdf(input_path, output_path):
    redact_hyperlinked_text(input_path, output_path)

In [2]:
def redact_hyperlinked_text(input_path, output_path):
    doc = fitz.open(input_path)
    
    for page in doc:
        # get_links() returns a list of dictionaries representing all clickable areas
        links = page.get_links()
        
        for link in links:
            # link["from"] contains the exact bounding box (fitz.Rect) of the hyperlink
            link_rect = link["from"]
            
            # Add a redaction annotation over this specific box, filled with white
            page.add_redact_annot(link_rect, fill=(1, 1, 1))
        
        # Apply the redactions, physically wiping the text under the boxes
        page.apply_redactions()
        
    doc.save(output_path)
    doc.close()
    print("All hyperlinked elements redacted. Clean PDF saved.")

In [31]:
def run_tests():
     # Locate the Data/tests directory
    test_dir = os.path.join("Data", "tests")
    if not os.path.exists(test_dir):
        test_dir = os.path.join("Data", "Tests")
        
    # Find all test PDF files
    pdf_files = glob.glob(os.path.join(test_dir, "*.pdf"))
    
    # Generate timestamp run directory (filesystem-friendly format: YYYY-MM-DD_HH-MM-SS)
    run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_dir = os.path.join(test_dir, "runs", run_id)
    
    clean_dir = os.path.join(run_dir, "clean")
    output_dir = os.path.join(run_dir, "output")
    
    os.makedirs(clean_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    
    for pdf_path in pdf_files:
        base_name = os.path.basename(pdf_path)
        base_no_ext = os.path.splitext(base_name)[0]
        
        # Configure output paths within the timestamped run directory
        output_path = os.path.join(output_dir, f"{base_no_ext}.md")
        clean_path = os.path.join(clean_dir, f"clean_{base_name}")
        
        print(f"Running test for {pdf_path} (Run: {run_id})...")
        produce_table_content(pdf_path, output_path, clean_pdf_output=clean_dir)

In [6]:
def get_tables(doc_str: str) -> list[str]:
    blocks = doc_str.split('\n\n')
    table_strings = []

    for i, block in enumerate(blocks):
        # A standard Markdown table contains a header separator row like "|---"
        if '|---' in block or '| ---' in block:
            # Add the table itself
            table_strings.append(f"**[Table Data]**\n{block.strip()}")
            table_strings.append("\n") # Add spacing between different table extractions

    # 3. Join the extracted blocks back into a single string
    return table_strings


def get_surrounding_paragraphs(doc_str: str, num_context_paragraphs: int = 1) -> list[tuple[str, str]]:
    blocks = doc_str.split('\n\n')
    surrounding_paragraphs = []

    for i, block in enumerate(blocks):
        # A standard Markdown table contains a header separator row like "|---"
        if '|---' in block or '| ---' in block:
            preceding_str = ""
            succeeding_str = ""
            
            # Grab paragraphs before the table (often the caption or introduction)
            preceding_paragraphs = []
            for j in range(1, num_context_paragraphs + 1):
                if i - j >= 0:
                    prev_block = blocks[i - j].strip()
                    if prev_block:
                        preceding_paragraphs.insert(0, prev_block)
            if preceding_paragraphs:
                preceding_text = "\n\n".join(preceding_paragraphs)
                preceding_str = f"**[Preceding Paragraphs]**\n{preceding_text}"
            

            # Grab paragraphs after the table (often table footnotes or continuation)
            succeeding_paragraphs = []
            for j in range(1, num_context_paragraphs + 1):
                if i + j < len(blocks):
                    next_block = blocks[i + j].strip()
                    if next_block:
                        succeeding_paragraphs.append(next_block)
            if succeeding_paragraphs:
                succeeding_text = "\n\n".join(succeeding_paragraphs)
                succeeding_str = f"**[Succeeding Paragraphs]**\n{succeeding_text}"


            surrounding_paragraphs.append((preceding_str, succeeding_str))

    return surrounding_paragraphs

In [20]:
def produce_table_content(input_path, output_path, clean_pdf_output=""):
    converter = DocumentConverter()
    
    # 1. Run conversion on raw input PDF to obtain context paragraphs
    raw_result = converter.convert(input_path)
    raw_markdown = raw_result.document.export_to_markdown()
    
    # 2. Set up the clean path (saving as clean+<filename> inside the "clean" subfolder)
    input_dir = os.path.dirname(input_path)
    filename = os.path.basename(input_path)
    clean_dir = os.path.join(input_dir, "clean")
    os.makedirs(clean_dir, exist_ok=True)
    
    clean_path = os.path.join(clean_pdf_output, f"clean_{filename}")

    
    # 3. Redact the hyperlinked text and save to clean_path
    redact_hyperlinked_text(input_path, clean_path)
    
    # 4. Convert the clean PDF
    parsed_result = converter.convert(clean_path)
    parsed_markdown = parsed_result.document.export_to_markdown()
    # 5. Extract tables and surrounding paragraphs
    tables = get_tables(parsed_markdown)
    surrounding_paragraphs = get_surrounding_paragraphs(raw_markdown, num_context_paragraphs=4)
    # Ensure parent output directory exists
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)


    # 6. Write out the markdown output in the same way as the test cell
    with open(output_path, 'w', encoding="utf-8") as output:
        # Prevent IndexError in case there are multiple tables
        limit = min(len(tables) - 1, len(surrounding_paragraphs))
        for i in range(limit):
            output.write(f"-------------- TABLE {i + 1} EXTRACTION --------------\n")
            output.write(f"{surrounding_paragraphs[i][0]}\n\n")
            output.write(f"{tables[i]}\n\n")
            output.write(surrounding_paragraphs[i][0])


    print(f"Written to file '{output_path}'")



In [32]:
# Test
# produce_table_content("./data/document.pdf", "./out.txt")

run_tests()

[INFO] 2026-06-23 16:14:09,164 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:09,171 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 16:14:09,171 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx


Running test for Data\tests\test1.pdf (Run: 2026-06-23_16-14-09)...


[INFO] 2026-06-23 16:14:09,324 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:09,326 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 16:14:09,328 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 16:14:09,422 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:09,433 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-06-23 16:14:09,434 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 7746.81it/s]


All hyperlinked elements redacted. Clean PDF saved.


[WARNING] 2026-06-23 16:14:33,748 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-06-23 16:14:34,163 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-06-23 16:14:34,782 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[INFO] 2026-06-23 16:14:45,642 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:45,649 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 16:14:45,650 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx


Written to file 'Data\tests\runs\2026-06-23_16-14-09\output\test1.md'
Running test for Data\tests\test2.pdf (Run: 2026-06-23_16-14-09)...


[INFO] 2026-06-23 16:14:45,832 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:45,834 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 16:14:45,835 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 16:14:45,911 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 16:14:45,922 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-06-23 16:14:45,923 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 7320.05it/s]
Stage preprocess failed for run 1, pages [8]: std::bad_alloc


All hyperlinked elements redacted. Clean PDF saved.


Stage preprocess failed for run 2, pages [8]: std::bad_alloc


Written to file 'Data\tests\runs\2026-06-23_16-14-09\output\test2.md'
